# 02 — Attention from Scratch

**Companion to [Chapter 04](../04-attention.md).**

Attention is the heart of the transformer. This notebook builds it in four stages,
each fixing a problem in the one before:

1. Similarity by dot product (no learning)
2. Queries, keys, values (adds learning)
3. Scaling by sqrt(d_k) (fixes saturation)
4. Causal masking (stops cheating)

Then multi-head attention, and a check against PyTorch's fused kernel.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(123)
torch.set_printoptions(precision=3, sci_mode=False)
print("torch", torch.__version__)

torch 2.13.0+cu130


## Stage 1 — Dot product = similarity

Three tokens, 2-dimensional embeddings so we can check the arithmetic by hand.

In [2]:
X = torch.tensor([[1., 0.],      # token 1
                  [0., 1.],      # token 2
                  [1., 1.]])     # token 3

scores = X @ X.T
print("attention scores (every token vs every token):")
print(scores)
print()
print("Row 3 is [1, 1, 2]: token 3 is most similar to itself, equally similar")
print("to tokens 1 and 2. That similarity IS the attention score.")

attention scores (every token vs every token):
tensor([[1., 0., 1.],
        [0., 1., 1.],
        [1., 1., 2.]])

Row 3 is [1, 1, 2]: token 3 is most similar to itself, equally similar
to tokens 1 and 2. That similarity IS the attention score.


Turn scores into weights with softmax, then use the weights to blend the vectors.

In [3]:
weights = torch.softmax(scores, dim=-1)
context = weights @ X

print("weights (each row sums to 1):")
print(weights)
print("\nrow sums:", weights.sum(dim=-1))
print("\ncontext vectors (weighted blends):")
print(context)

weights (each row sums to 1):
tensor([[0.422, 0.155, 0.422],
        [0.155, 0.422, 0.422],
        [0.212, 0.212, 0.576]])

row sums: tensor([1., 1., 1.])

context vectors (weighted blends):
tensor([[0.845, 0.578],
        [0.578, 0.845],
        [0.788, 0.788]])


## Stage 2 — Queries, Keys, Values

Comparing raw embeddings gives the model no control. Project each token three ways:

| | Question |
|---|---|
| **Query** | what am I looking for? |
| **Key** | what do I offer? |
| **Value** | what do I contribute if selected? |

These three matrices are what training learns.

In [4]:
d_in, d_out = 2, 4

W_query = nn.Linear(d_in, d_out, bias=False)
W_key   = nn.Linear(d_in, d_out, bias=False)
W_value = nn.Linear(d_in, d_out, bias=False)

Q, K, V = W_query(X), W_key(X), W_value(X)
print("Q shape:", Q.shape, " K shape:", K.shape, " V shape:", V.shape)

scores = Q @ K.T
print("\nscores now come from LEARNED projections:")
print(scores)

Q shape: torch.Size([3, 4])  K shape: torch.Size([3, 4])  V shape: torch.Size([3, 4])

scores now come from LEARNED projections:
tensor([[ 0.716,  0.145,  0.860],
        [-0.245, -0.066, -0.311],
        [ 0.470,  0.078,  0.549]], grad_fn=<MmBackward0>)


## Stage 3 — Why divide by sqrt(d_k)

As dimension grows, dot products grow, and large scores saturate softmax into a
near one-hot distribution — where the gradient is ~0 and **learning stops**.

Run this to see saturation happen.

In [5]:
for d_k in [4, 64, 1024]:
    a = torch.randn(d_k)
    b = torch.randn(d_k)
    raw = (a @ b).item()
    scaled = raw / d_k ** 0.5
    print(f"d_k={d_k:5d}   raw dot product {raw:9.3f}   scaled {scaled:7.3f}")

print()
print("softmax([1, 2, 3])     =", torch.softmax(torch.tensor([1., 2., 3.]), 0).tolist())
print("softmax([10, 20, 30])  =", torch.softmax(torch.tensor([10., 20., 30.]), 0).tolist())
print()
print("The second is saturated: one weight is 1.0, the rest are 0.")
print("Gradients there are effectively zero, so the model cannot learn.")

d_k=    4   raw dot product    -1.180   scaled  -0.590
d_k=   64   raw dot product    -2.152   scaled  -0.269
d_k= 1024   raw dot product    64.335   scaled   2.010

softmax([1, 2, 3])     = [0.09003057330846786, 0.2447284758090973, 0.6652409434318542]
softmax([10, 20, 30])  = [2.0610599893444714e-09, 4.539786823443137e-05, 0.9999545812606812]

The second is saturated: one weight is 1.0, the rest are 0.
Gradients there are effectively zero, so the model cannot learn.


## Stage 4 — Causal masking

A language model must not see the future. Mask scores where `column > row`
with `-inf` **before** softmax, because `exp(-inf) = 0` exactly, so the
remaining weights still sum to 1 with no renormalization needed.

In [6]:
n = 3
mask = torch.triu(torch.ones(n, n), diagonal=1).bool()
print("mask (True = block):")
print(mask)

X = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
d_k = 2

scores = X @ X.T / d_k ** 0.5
print("\nscaled scores:")
print(scores)

scores = scores.masked_fill(mask, float('-inf'))
print("\nmasked:")
print(scores)

weights = torch.softmax(scores, dim=-1)
context = weights @ X
print("\nweights:")
print(weights)
print("\ncontext:")
print(context)

mask (True = block):
tensor([[False,  True,  True],
        [False, False,  True],
        [False, False, False]])

scaled scores:
tensor([[0.707, 0.000, 0.707],
        [0.000, 0.707, 0.707],
        [0.707, 0.707, 1.414]])

masked:
tensor([[0.707,  -inf,  -inf],
        [0.000, 0.707,  -inf],
        [0.707, 0.707, 1.414]])

weights:
tensor([[1.000, 0.000, 0.000],
        [0.330, 0.670, 0.000],
        [0.248, 0.248, 0.503]])

context:
tensor([[1.000, 0.000],
        [0.330, 0.670],
        [0.752, 0.752]])


### Verify the hand calculation

Chapter 04 works this exact example out on paper. The weights should be:

```
row 1: [1.000, 0.000, 0.000]
row 2: [0.330, 0.670, 0.000]
row 3: [0.248, 0.248, 0.503]
```

In [7]:
expected = torch.tensor([[1.000, 0.000, 0.000],
                         [0.330, 0.670, 0.000],
                         [0.248, 0.248, 0.503]])
print("computed:\n", weights)
print("\nmatches hand calculation:",
      torch.allclose(weights, expected, atol=1e-3))

computed:
 tensor([[1.000, 0.000, 0.000],
        [0.330, 0.670, 0.000],
        [0.248, 0.248, 0.503]])

matches hand calculation: True


## Multi-head attention

One attention operation learns one notion of relevance. Run several in parallel
on different slices of the representation and you get several.

The efficient implementation does **not** loop over heads — it reshapes one big
projection into a head dimension.

In [8]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout=0.0,
                 num_heads=12, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)

        # buffer: moves with .to(device), saved in state_dict, but NOT trained
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x, verbose=False):
        b, num_tokens, _ = x.shape

        queries = self.W_query(x)
        keys    = self.W_key(x)
        values  = self.W_value(x)
        if verbose:
            print(f"  after projection      {tuple(queries.shape)}")

        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys    = keys.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values  = values.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        if verbose:
            print(f"  after head split      {tuple(queries.shape)}")

        attn_scores = queries @ keys.transpose(2, 3) / keys.shape[-1] ** 0.5
        if verbose:
            print(f"  attention matrix      {tuple(attn_scores.shape)}  <-- seq x seq, QUADRATIC")

        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores = attn_scores.masked_fill(mask_bool, -torch.inf)

        attn_weights = self.dropout(torch.softmax(attn_scores, dim=-1))
        context = attn_weights @ values
        if verbose:
            print(f"  after weighted sum    {tuple(context.shape)}")

        context = context.transpose(1, 2).contiguous().view(b, num_tokens, self.d_out)
        if verbose:
            print(f"  after merging heads   {tuple(context.shape)}")
        return self.out_proj(context)


torch.manual_seed(123)
mha = MultiHeadAttention(d_in=768, d_out=768, context_length=1024, num_heads=12)
x = torch.randn(2, 6, 768)

print("shape walkthrough (batch=2, seq=6, d=768, heads=12, head_dim=64):")
out = mha(x, verbose=True)
print(f"\noutput {tuple(out.shape)}")
print(f"parameters: {sum(p.numel() for p in mha.parameters()):,}")

shape walkthrough (batch=2, seq=6, d=768, heads=12, head_dim=64):
  after projection      (2, 6, 768)
  after head split      (2, 12, 6, 64)
  attention matrix      (2, 12, 6, 6)  <-- seq x seq, QUADRATIC
  after weighted sum    (2, 12, 6, 64)
  after merging heads   (2, 6, 768)

output (2, 6, 768)
parameters: 2,360,064


## The quadratic cost, made concrete

The attention matrix is `(seq, seq)`. Doubling the context quadruples it.
Every long-context innovation in the field attacks this number.

In [9]:
print(f"{'context':>10} {'attn matrix elements':>22} {'memory (bf16, 12 heads)':>26}")
for seq in [512, 1024, 4096, 16384, 65536]:
    elements = seq * seq * 12
    gb = elements * 2 / 1e9
    print(f"{seq:>10,} {elements:>22,} {gb:>23.2f} GB")

print("\nThis is per layer, per sequence in the batch.")
print("It is also exactly what FlashAttention avoids materializing.")

   context   attn matrix elements    memory (bf16, 12 heads)
       512              3,145,728                    0.01 GB
     1,024             12,582,912                    0.03 GB
     4,096            201,326,592                    0.40 GB
    16,384          3,221,225,472                    6.44 GB
    65,536         51,539,607,552                  103.08 GB

This is per layer, per sequence in the batch.
It is also exactly what FlashAttention avoids materializing.


## Check against PyTorch's fused kernel

`F.scaled_dot_product_attention` computes the same thing but never materializes
the score matrix — it tiles the computation in fast on-chip memory
(FlashAttention). Same math, far less memory traffic.

In [10]:
torch.manual_seed(0)
b, h, s, hd = 2, 4, 16, 8
q, k, v = torch.randn(b, h, s, hd), torch.randn(b, h, s, hd), torch.randn(b, h, s, hd)

# manual
scores = q @ k.transpose(-2, -1) / hd ** 0.5
causal = torch.triu(torch.ones(s, s), diagonal=1).bool()
manual = torch.softmax(scores.masked_fill(causal, -torch.inf), dim=-1) @ v

# fused
fused = F.scaled_dot_product_attention(q, k, v, is_causal=True)

print("identical:", torch.allclose(manual, fused, atol=1e-5))
print("max difference:", (manual - fused).abs().max().item())
print("\nWrite the manual version once to understand it. Use the fused one in real code.")

identical: True
max difference: 5.960464477539062e-07

Write the manual version once to understand it. Use the fused one in real code.


## Exercises

1. Set `num_heads=5` with `d_out=768` and read the assertion error. Why must they divide evenly?
2. Delete the `masked_fill` line and re-run. What happens to row 1's weights, and why would training loss collapse?
3. Remove the `/ keys.shape[-1] ** 0.5` scaling and inspect the weights at `d_out=768`. How saturated are they?

**Next:** [03 — Build a GPT](03_build_gpt.ipynb)